# Experiment: AI x India Landscape

This notebook builds a mini Morgan Stanley-style thematic pitch around listed Indian AI, analytics, and IT-enablement equities. It uses live market data for returns, risk, and basic fundamentals, then exports a slide-ready markdown deck into `reports/ai_india_thematic_pitch.md`.

In [ ]:
from pathlib import Path
import logging
import sys

import pandas as pd
import plotly.io as pio
from IPython.display import Markdown, display

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / 'src').exists() else NOTEBOOK_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.analysis import (
    DEFAULT_BENCHMARK_TICKER,
    build_ai_india_dataset,
    build_hypothetical_ideas,
    cache_stats_path,
    indexed_price_frame,
    make_indexed_performance_chart,
    make_return_bar_chart,
    make_valuation_scatter,
)
from src.reporting import build_thematic_pitch_markdown, company_snapshot_table, ranking_table, write_markdown
from src.scoring import compute_factor_scores

logging.basicConfig(level=logging.WARNING, format='%(levelname)s | %(message)s')
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)
pio.renderers.default = 'notebook_connected'


In [ ]:
START_DATE = '2021-01-01'
REPRESENTATIVE_TICKERS = ['INFY.NS', 'PERSISTENT.NS', 'TATAELXSI.NS', 'NETWEB.NS']

universe, price_history, benchmark_history, stats = build_ai_india_dataset(
    start=START_DATE,
    benchmark_ticker=DEFAULT_BENCHMARK_TICKER,
)
scored_stats = compute_factor_scores(stats)
stats_path = cache_stats_path()
scored_stats.to_csv(stats_path, index=False)

print(f'Loaded {len(universe)} companies and {len(price_history):,} price rows.')
print(f'Cached latest stats to: {stats_path}')
scored_stats.head()


In [ ]:
snapshot_table = company_snapshot_table(scored_stats.sort_values('market_cap', ascending=False))

top_3y = ranking_table(
    scored_stats.nlargest(5, 'cagr_3y'),
    ['name', 'segment', 'cagr_3y', 'return_1y', 'market_cap'],
).rename(columns={'name': 'Company', 'segment': 'Segment', 'cagr_3y': '3Y CAGR', 'return_1y': '1Y Return', 'market_cap': 'Market Cap'})

top_mcap = ranking_table(
    scored_stats.nlargest(5, 'market_cap'),
    ['name', 'segment', 'market_cap', 'trailing_pe', 'dividend_yield'],
).rename(columns={'name': 'Company', 'segment': 'Segment', 'market_cap': 'Market Cap', 'trailing_pe': 'Trailing PE', 'dividend_yield': 'Dividend Yield'})

top_ai = ranking_table(
    scored_stats.nlargest(5, 'ai_purity_score'),
    ['name', 'segment', 'ai_purity_score', 'cagr_3y', 'market_cap'],
).rename(columns={'name': 'Company', 'segment': 'Segment', 'ai_purity_score': 'AI Purity Score', 'cagr_3y': '3Y CAGR', 'market_cap': 'Market Cap'})

display(snapshot_table)
display(top_3y)
display(top_mcap)
display(top_ai)


In [ ]:
label_map = {
    'INFY.NS': 'Infosys',
    'PERSISTENT.NS': 'Persistent Systems',
    'TATAELXSI.NS': 'Tata Elxsi',
    'NETWEB.NS': 'Netweb',
    DEFAULT_BENCHMARK_TICKER: 'NIFTY 50',
}

indexed_frame = indexed_price_frame(
    price_history=price_history,
    benchmark_history=benchmark_history,
    tickers=REPRESENTATIVE_TICKERS,
    label_map=label_map,
)

indexed_fig = make_indexed_performance_chart(indexed_frame, 'AI x India: Representative Indexed Performance vs. NIFTY 50')
bar_fig = make_return_bar_chart(scored_stats, 'AI x India Universe: 1Y Total Return')
scatter_fig = make_valuation_scatter(scored_stats)

indexed_fig.show()
bar_fig.show()
scatter_fig.show()


## Macro and thematic context

- IBEF's March 2026 IT & BPM update says India's IT industry is likely to hit **US$350 billion by 2026** and highlights BFSI, telecom, and retail as core verticals. That supports a broad AI-adoption runway for listed Indian technology providers rather than a narrow single-sector trade.
- The same IBEF note says India's **AI market could reach US$28.8 billion by 2025 at a 45% CAGR**, reinforcing why engineering-R&D, data, and enterprise-software specialists are showing up in public equity screens.
- PwC India said in July 2025 that businesses could unlock **US$9.82 trillion of GVA by 2035** by aligning with fast-changing human and industrial domains, which frames AI as a cross-sector productivity theme spanning services, manufacturing, and enterprise workflows.

Public links used in the report export are embedded in the generated markdown file.

In [ ]:
top_cagr_names = scored_stats.nlargest(3, 'cagr_3y')['name'].tolist()
top_market_caps = scored_stats.nlargest(3, 'market_cap')['name'].tolist()

executive_summary = [
    f'The screened AI x India universe spans {len(scored_stats)} listed names across IT services, ER&D, enterprise software, analytics platforms, ad-tech, and AI infrastructure.',
    f"Market-cap depth is anchored by {', '.join(top_market_caps)}, keeping the theme investable for both wealth and capital-markets conversations.",
    f"Trailing performance leadership is currently concentrated in {', '.join(top_cagr_names)}, highlighting how alpha has skewed toward specialist engineering, analytics, and platform plays.",
    'Valuation dispersion remains wide, so the theme is best framed as a barbell between core large-cap IT enablers and higher-beta AI specialists.',
]

macro_context = [
    "IBEF's March 2026 IT & BPM update says India's IT industry is likely to hit US$350bn by 2026 and identifies BFSI, telecom, and retail as core verticals, supporting a broad AI adoption runway for listed Indian technology providers. Source: [IBEF IT & BPM Industry in India](https://www.ibef.org/industry/information-technology-india).",
    "The same IBEF note says India's AI market is projected to reach US$28.8bn by 2025 at a 45% CAGR and highlights GenAI plus engineering R&D as demand catalysts, which is particularly relevant for ER&D and digital-engineering names in this screen. Source: [IBEF IT & BPM Industry in India](https://www.ibef.org/industry/information-technology-india).",
    "PwC India said in July 2025 that businesses can unlock US$9.82tn of GVA by 2035 by aligning with fast-changing human and industrial domains, framing AI as a cross-sector productivity theme rather than a single-software niche. Source: [PwC India - Navigating the value shift](https://www.pwc.in/press-releases/2025/domains-driven-by-human-and-industrial-needs-can-unlock-usd-982-trillion-in-growth-opportunities-by-2035-pwc-india-report.html).",
]

risks = [
    'Many listed beneficiaries are still discretionary IT-spending names, so a global slowdown or project deferrals can hit growth and sentiment together.',
    'Thematic multiples can compress quickly when GenAI enthusiasm outruns near-term revenue realization, especially for higher-beta mid-caps.',
    'Some businesses are broad IT or industrial conglomerates rather than pure-play AI exposures, so security selection matters more than the headline theme.',
    'Public APIs are appropriate for a demo, but real banking workflows would need licensed data feeds, corporate-action controls, and compliance-approved content pipelines.',
]

slide_outline = [
    'Slide 1 - Theme overview: why AI matters for India equities and advisory conversations.',
    'Slide 2 - Performance and valuation snapshot of the listed AI India universe.',
    'Slide 3 - Key listed winners, ranked by size, performance, and AI purity.',
    'Slide 4 - Hypothetical M&A and partnership ideas across AI infrastructure, analytics, travel-tech, and mobility software.',
]

sources = [
    '[IBEF - IT & BPM Industry in India](https://www.ibef.org/industry/information-technology-india)',
    '[PwC India - Navigating the value shift](https://www.pwc.in/press-releases/2025/domains-driven-by-human-and-industrial-needs-can-unlock-usd-982-trillion-in-growth-opportunities-by-2035-pwc-india-report.html)',
]

report_text = build_thematic_pitch_markdown(
    as_of_date=scored_stats['as_of_date'].iloc[0],
    executive_summary=executive_summary,
    macro_context=macro_context,
    company_snapshot=snapshot_table,
    ranking_sections={
        'Top 5 by 3Y CAGR': top_3y,
        'Top 5 by Market Cap': top_mcap,
        'Top 5 by AI Purity': top_ai,
    },
    risks=risks,
    hypothetical_ideas=build_hypothetical_ideas(scored_stats),
    slide_outline=slide_outline,
    sources=sources,
)

report_path = write_markdown('reports/ai_india_thematic_pitch.md', report_text)
display(Markdown(f'Report written to `{report_path}`'))
display(Markdown(report_text[:2500]))
